# Finnish ASR Baseline: wav2vec 2.0 + CTC

Team Tokens — COE025 Natural Language Processing, Project 3.

This experiment evaluates the published Finnish ASR checkpoint `jonatasgrosman/wav2vec2-large-xlsr-53-finnish` without additional training by the team. It is a pretrained baseline, not a team fine-tuning contribution.

**Checkpoint revision:** `a497f86c265cffe9f5d6f0162bc7862dc3e11c00`.

The [model card](https://huggingface.co/jonatasgrosman/wav2vec2-large-xlsr-53-finnish) reports training on Finnish Common Voice 6.1 and CSS10. Recording-level and speaker-level overlap with the team's Common Voice 27 test set has not been verified.

The recorded run used a Tesla T4 GPU, float32 precision, and a batch size of 4. Decoding uses greedy CTC without an external language model.


## 1. Dependencies


In [1]:
%pip -q install transformers==4.57.6 jiwer==4.0.0 soundfile==0.13.1 librosa==0.11.0 pyarrow huggingface-hub


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.


## 2. Experiment Files

The embedded archive contains the evaluation scripts, fixed data manifests, and experiment protocol. Its checksum is verified before extraction.


In [2]:
import base64,hashlib,io,zipfile
from pathlib import Path
PACKAGE=Path('/content/ece_wav2vec2')
blob=base64.b64decode('...')
assert hashlib.sha256(blob).hexdigest()=='4eeed7bade4cb7517fc2d5ad194da22aaa22595c81ca4c9ee5655b5148a098ad'
PACKAGE.mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(io.BytesIO(blob)) as archive:
    assert all((PACKAGE/m.filename).resolve().is_relative_to(PACKAGE.resolve()) for m in archive.infolist())
    archive.extractall(PACKAGE)
print('Dosyalar doğrulandı.')


Dosyalar doğrulandı.


## 3. Environment and Data Validation

Checks cover GPU availability, available disk space, the shared evaluator, manifest integrity, and reference normalization.


In [3]:
import os,sys,subprocess,datetime,platform,shutil
from collections import deque
import torch
from packaging.version import Version
OUT=Path('/content/Ece_Wav2Vec2_outputs')
OUT.mkdir(parents=True,exist_ok=True)
CACHE='/content/asr_audio'
os.environ['HF_HUB_DISABLE_XET']='1'
os.environ['HF_HUB_DISABLE_TELEMETRY']='1'
assert torch.cuda.is_available(), 'Çalışma zamanı türünü değiştir → T4 GPU seç.'
assert Version(torch.__version__.split('+')[0])>=Version('2.6'), 'Bu checkpoint için torch >=2.6 gerekli; ortamı kontrol edelim.'
print('GPU:',torch.cuda.get_device_name(0),'| Python:',platform.python_version())
print('Boş disk:',round(shutil.disk_usage('/content').free/2**30,1),'GB')
assert shutil.disk_usage('/content').free>12*2**30, 'Yeterli boş disk alanı yok.'
def command(args):
    logs=OUT/'logs';logs.mkdir(exist_ok=True)
    log_path=logs/(datetime.datetime.now().strftime('%Y%m%d_%H%M%S_%f')+'.log')
    tail=deque(maxlen=40)
    with log_path.open('w',encoding='utf8') as log:
        with subprocess.Popen(list(map(str,args)),stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,
                              encoding='utf8',errors='replace',bufsize=1) as process:
            for line in process.stdout:
                print(line,end='',flush=True);log.write(line);log.flush();tail.append(line)
            status=process.wait()
    if status:
        print('SON HATA SATIRLARI:\n'+''.join(tail))
        raise RuntimeError(f'İşlem durdu. Son hata satırlarını paylaş. Kayıt: {log_path}')
def run(script,*args):command([sys.executable,'-u',PACKAGE/script,*args])
run('test_protocol.py')
print('HAZIRLIK TAMAM')


GPU: Tesla T4 | Python: 3.13.16
Boş disk: 112.4 GB
...
----------------------------------------------------------------------
Ran 3 tests in 0.252s

OK
HAZIRLIK TAMAM


## 4. Development Smoke Test

The sample consists of the first eight and the eight longest recordings in `dev_cv`, with duplicates removed. It checks the inference pipeline, including longer inputs. These sample scores are not full development-set or test-set results.

Runtime estimates extrapolate the measured inference time and exclude downloads and setup.


In [4]:
run('eval_wav2vec2.py','--split','dev_cv','--out',OUT,'--cache',CACHE,'--smoke')


I0000 00:00:1795258150.668927    1791 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1795258153.544618    1791 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
Model yükleniyor: jonatasgrosman/wav2vec2-large-xlsr-53-finnish
/usr/local/lib/python3.13/dist-packages/transformers/configuration_utils.py:335: UserWarning: Passing `gradient_checkpointing` to a config initialization is deprecated and will be removed in v5 Transformers. Using `model.gradient_checkpointing_enable()` instead, or if you are using the `Trainer` API, pass `gradient_checkpointing=True` in your `TrainingArguments`.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!
dev_cv: 16/16 kayıt tamamlandı
REF: punaisessa mekossa 
HYP: punaisessamekos

## 5. Result Archive

Each archive contains the results available at the time of export, including predictions, scores, logs, and the experiment scripts. Audio files and model weights are excluded.


In [5]:
from google.colab import files
def download_results():
    path=Path('/content/Ece_Wav2Vec2_sonuclar_'+datetime.datetime.now().strftime('%Y%m%d_%H%M%S')+'.zip')
    with zipfile.ZipFile(path,'w',zipfile.ZIP_DEFLATED) as z:
        for source,prefix in [(OUT,''),(PACKAGE,'scripts/ece_wav2vec2')]:
            for p in sorted(source.rglob('*')):
                if p.is_file() and '__pycache__' not in p.parts and p.suffix in {'.py','.json','.jsonl','.csv','.md','.log'}:
                    z.write(p,Path(prefix)/p.relative_to(source))
    print('İndiriliyor:',path)
    files.download(str(path))
download_results()


İndiriliyor: /content/Ece_Wav2Vec2_sonuclar_20261121_105352.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 6. Common Voice Test Set — `test_cv`

The test set contains **1,833 recordings**. Predictions are matched to the fixed manifest by recording ID.


In [6]:
assert (OUT/'dev_smoke/scores/wav2vec2-xlsr-53-finnish-ctc-baseline.csv').exists(), 'Önce kısa deneme tamamlanmalı.'
run('eval_wav2vec2.py','--split','test_cv','--out',OUT,'--cache',CACHE)


I0000 00:00:1795258445.413537    3073 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1795258448.274339    3073 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
Model yükleniyor: jonatasgrosman/wav2vec2-large-xlsr-53-finnish
/usr/local/lib/python3.13/dist-packages/transformers/configuration_utils.py:335: UserWarning: Passing `gradient_checkpointing` to a config initialization is deprecated and will be removed in v5 Transformers. Using `model.gradient_checkpointing_enable()` instead, or if you are using the `Trainer` API, pass `gradient_checkpointing=True` in your `TrainingArguments`.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!
test_cv: 40/1833 kayıt tamamlandı
test_cv: 80/1833 kayıt tamamlandı
test_cv:

In [7]:
download_results()


İndiriliyor: /content/Ece_Wav2Vec2_sonuclar_20261121_105719.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 7. Parliament Test Set — `test_parl`

The test set contains **1,913 recordings**. Predictions are matched to the fixed manifest by recording ID.


In [8]:
assert (OUT/'dev_smoke/scores/wav2vec2-xlsr-53-finnish-ctc-baseline.csv').exists(), 'Önce kısa deneme tamamlanmalı.'
run('eval_wav2vec2.py','--split','test_parl','--out',OUT,'--cache',CACHE)


I0000 00:00:1795258785.885232    4544 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1795258788.924612    4544 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
Model yükleniyor: jonatasgrosman/wav2vec2-large-xlsr-53-finnish
/usr/local/lib/python3.13/dist-packages/transformers/configuration_utils.py:335: UserWarning: Passing `gradient_checkpointing` to a config initialization is deprecated and will be removed in v5 Transformers. Using `model.gradient_checkpointing_enable()` instead, or if you are using the `Trainer` API, pass `gradient_checkpointing=True` in your `TrainingArguments`.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!
test_parl: 40/1913 kayıt tamamlandı
test_parl: 80/1913 kayıt tamamlandı
test

In [9]:
download_results()


İndiriliyor: /content/Ece_Wav2Vec2_sonuclar_20261121_110852.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 8. FLEURS Test Set — `test_fleurs`

The test set contains **916 recordings**. Predictions are matched to the fixed manifest by recording ID.


In [10]:
assert (OUT/'dev_smoke/scores/wav2vec2-xlsr-53-finnish-ctc-baseline.csv').exists(), 'Önce kısa deneme tamamlanmalı.'
run('eval_wav2vec2.py','--split','test_fleurs','--out',OUT,'--cache',CACHE)


I0000 00:00:1795259401.333195    7166 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1795259403.898557    7166 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
Model yükleniyor: jonatasgrosman/wav2vec2-large-xlsr-53-finnish
/usr/local/lib/python3.13/dist-packages/transformers/configuration_utils.py:335: UserWarning: Passing `gradient_checkpointing` to a config initialization is deprecated and will be removed in v5 Transformers. Using `model.gradient_checkpointing_enable()` instead, or if you are using the `Trainer` API, pass `gradient_checkpointing=True` in your `TrainingArguments`.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!
test_fleurs: 40/916 kayıt tamamlandı
test_fleurs: 80/916 kayıt tamamlandı
te

In [11]:
download_results()


İndiriliyor: /content/Ece_Wav2Vec2_sonuclar_20261121_111457.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 9. Results and Evaluation Notes

All three test sets were completed in the recorded run. Each test set is reported separately.

| Test set | Recordings | WER (%) | CER (%) |
|---|---:|---:|---:|
| Common Voice (`test_cv`) | 1,833 | 42.78 | 8.93 |
| Parliament (`test_parl`) | 1,913 | 56.22 | 13.07 |
| FLEURS (`test_fleurs`) | 916 | 59.52 | 15.83 |

Scores and per-recording predictions are exported in the team's standard `results/scores` and `results/predictions` formats.

References use the manifest's existing `text` values. The shared `asr_eval.py` normalizer is applied once to each raw model prediction. Corpus-level WER and CER are computed with the same evaluator used by the other systems.

Inference timing includes preprocessing, GPU transfer, the forward pass, and CTC text decoding after a development warm-up, with CUDA synchronization. WAV reading, downloads, model loading, scoring, and logging are excluded. FP32/batch-4 speed measurements are not a controlled comparison with FP16/batch-16 Whisper or Qwen runs.

The checkpoint's prior Common Voice training and unverified overlap with the current evaluation data remain limitations of this baseline.
